In [0]:
df_acidentes_bronze = spark.table("workspace.bronze.acidentes")

print("Tabela Bronze de acidentes carregada com sucesso.")

In [0]:
display(df_acidentes_bronze.limit(5))

In [0]:
# Amostra dos registros do município do Rio de Janeiro
df_rio_amostra = (
    df_acidentes_bronze
    .filter("codigo_ibge = '3304557'")
    .select(
        "num_acidente",
        "data_acidente",
        "uf_acidente",
        "codigo_ibge",
        "bairro_acidente",
        "dia_semana",
        "hora_acidente"
    )
    .limit(10)
)

display(df_rio_amostra)

In [0]:
df_marcadores_bairro = (
    df_acidentes_bronze
    .filter(F.col("codigo_ibge") == "3304557")
    .filter(F.col("bairro_acidente").isNotNull())
    .filter(
        F.upper(F.trim(F.col("bairro_acidente"))).isin(
            "NI",
            "NAO INFORMADO",
            "NÃO INFORMADO",
            "DESCONHECIDO",
            "IGNORADO",
            "SEM INFORMACAO",
            "SEM INFORMAÇÃO"
        )
    )
    .groupBy("bairro_acidente")
    .count()
    .orderBy(F.desc("count"))
)

display(df_marcadores_bairro)

In [0]:
df_hora_formato = (
    df_acidentes_bronze
    .filter(F.col("codigo_ibge") == "3304557")
    .select(
        F.length(F.trim(F.col("hora_acidente"))).alias("tamanho"),
        "hora_acidente"
    )
    .groupBy("tamanho")
    .agg(
        F.count("*").alias("quantidade"),
        F.min("hora_acidente").alias("exemplo_min"),
        F.max("hora_acidente").alias("exemplo_max")
    )
    .orderBy("tamanho")
)

display(df_hora_formato)

In [0]:
df_hora_validacao = (
    df_acidentes_bronze
    .filter(F.col("codigo_ibge") == "3304557")
    .select(
        "hora_acidente",
        F.substring("hora_acidente", 1, 2).cast("int").alias("hora"),
        F.substring("hora_acidente", 3, 2).cast("int").alias("minuto"),
        F.substring("hora_acidente", 5, 2).cast("int").alias("segundo")
    )
    .filter(
        (F.col("hora") > 23) |
        (F.col("minuto") > 59) |
        (F.col("segundo") > 59)
    )
)

display(df_hora_validacao.limit(20))

In [0]:
df_data_validacao = (
    df_acidentes_bronze
    .filter(F.col("codigo_ibge") == "3304557")
    .select(
        "data_acidente",
        F.to_date("data_acidente", "yyyy-MM-dd").alias("data_convertida")
    )
    .filter(
        F.col("data_acidente").isNotNull() &
        F.col("data_convertida").isNull()
    )
)

display(df_data_validacao.limit(20))

In [0]:
df_duplicidades = (
    df_acidentes_bronze
    .filter(F.col("codigo_ibge") == "3304557")
    .groupBy("num_acidente")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

display(df_duplicidades.limit(20))

In [0]:
campos_quantitativos = [
    "qtde_acidente",
    "qtde_envolvidos",
    "qtde_feridosilesos",
    "qtde_obitos"
]

for campo in campos_quantitativos:
    invalidos = (
        df_acidentes_bronze
        .filter(F.col("codigo_ibge") == "3304557")
        .filter(
            F.col(campo).isNotNull() &
            F.col(campo).cast("int").isNull()
        )
        .count()
    )

    print(f"{campo}: {invalidos} valores não convertíveis para inteiro")

In [0]:
# Construção da camada Silver de Acidentes
df_acidentes_silver = (
    df_acidentes_bronze

    # Recorte geográfico: município do Rio de Janeiro
    .filter(F.col("codigo_ibge") == "3304557")

    # Tipagem e padronização
    .withColumn(
        "data_acidente",
        F.to_date(F.col("data_acidente"), "yyyy-MM-dd")
    )
    .withColumn(
        "hora_acidente",
        F.to_timestamp(
            F.concat(
                F.col("data_acidente").cast("string"),
                F.lit(" "),
                F.col("hora_acidente")
            ),
            "yyyy-MM-dd HHmmss"
        )
    )

    # Padronização de bairro
    .withColumn(
        "bairro_acidente",
        F.when(
            F.upper(F.trim(F.col("bairro_acidente"))).isin(
                "NI",
                "SEM INFORMACAO"
            ),
            F.lit(None)
        ).otherwise(
            F.upper(F.trim(F.col("bairro_acidente")))
        )
    )

    # Tipagem dos campos quantitativos
    .withColumn("qtde_acidente", F.col("qtde_acidente").cast("int"))
    .withColumn("qtde_envolvidos", F.col("qtde_envolvidos").cast("int"))
    .withColumn("qtde_feridosilesos", F.col("qtde_feridosilesos").cast("int"))
    .withColumn("qtde_obitos", F.col("qtde_obitos").cast("int"))
)

print("DataFrame Silver de acidentes construído com sucesso.")

In [0]:
df_validacao_silver = (
    df_acidentes_silver
    .select(
        "num_acidente",
        "codigo_ibge",
        "data_acidente",
        "hora_acidente",
        "bairro_acidente",
        "qtde_acidente",
        "qtde_envolvidos",
        "qtde_feridosilesos",
        "qtde_obitos"
    )
    .limit(10)
)

display(df_validacao_silver)

print("\nSchema Silver:")
df_acidentes_silver.select(
    "data_acidente",
    "hora_acidente",
    "qtde_acidente",
    "qtde_envolvidos",
    "qtde_feridosilesos",
    "qtde_obitos"
).printSchema()

In [0]:
df_validacao_bairro = (
    df_acidentes_silver
    .filter(
        F.upper(F.trim(F.col("bairro_acidente"))).isin(
            "NI",
            "SEM INFORMACAO"
        )
    )
)

print(
    "Marcadores de ausência restantes:",
    df_validacao_bairro.count()
)

In [0]:
# Persistência da tabela Acidentes na camada Silver
(
    df_acidentes_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.acidentes")
)

print("Tabela workspace.silver.acidentes gravada com sucesso.")

In [0]:
df_localidade_bronze = spark.table("workspace.bronze.localidade")

print("Tabela Bronze de localidade carregada com sucesso.")

display(
    df_localidade_bronze
    .filter(F.col("codigo_ibge") == "3304557")
    .limit(10)
)

In [0]:
df_localidade_rio = (
    df_localidade_bronze
    .filter(F.col("codigo_ibge") == "3304557")
    .select(
        "ano_referencia",
        "mes_referencia",
        "municipio",
        "qtde_habitantes",
        "frota_total",
        "frota_circulante"
    )
    .orderBy("ano_referencia", "mes_referencia")
)

display(df_localidade_rio)

In [0]:
campos_localidade = [
    "qtde_habitantes",
    "frota_total",
    "frota_circulante"
]

for campo in campos_localidade:
    invalidos = (
        df_localidade_bronze
        .filter(F.col("codigo_ibge") == "3304557")
        .filter(
            F.col(campo).isNotNull() &
            F.col(campo).cast("long").isNull()
        )
        .count()
    )

    print(f"{campo}: {invalidos} valores não convertíveis para inteiro")

In [0]:
# Construção da camada Silver de Localidade
df_localidade_silver = (
    df_localidade_bronze

    # Recorte geográfico: município do Rio de Janeiro
    .filter(F.col("codigo_ibge") == "3304557")

    # Tipagem dos campos temporais
    .withColumn(
        "ano_referencia",
        F.col("ano_referencia").cast("int")
    )
    .withColumn(
        "mes_referencia",
        F.col("mes_referencia").cast("int")
    )

    # Tipagem dos indicadores
    .withColumn(
        "qtde_habitantes",
        F.col("qtde_habitantes").cast("long")
    )
    .withColumn(
        "frota_total",
        F.col("frota_total").cast("long")
    )
    .withColumn(
        "frota_circulante",
        F.col("frota_circulante").cast("long")
    )

    # Padronização textual
    .withColumn(
        "municipio",
        F.upper(F.trim(F.col("municipio")))
    )
    .withColumn(
        "uf",
        F.upper(F.trim(F.col("uf")))
    )
)

print("DataFrame Silver de localidade construído com sucesso.")

In [0]:
display(
    df_localidade_silver
    .select(
        "chv_localidade",
        "ano_referencia",
        "mes_referencia",
        "codigo_ibge",
        "municipio",
        "uf",
        "qtde_habitantes",
        "frota_total",
        "frota_circulante"
    )
    .orderBy("ano_referencia", "mes_referencia")
    .limit(10)
)

print("\nSchema Silver - Localidade:")

df_localidade_silver.select(
    "ano_referencia",
    "mes_referencia",
    "qtde_habitantes",
    "frota_total",
    "frota_circulante"
).printSchema()

In [0]:
# Persistência da tabela Localidade na camada Silver
(
    df_localidade_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.localidade")
)

print("Tabela workspace.silver.localidade gravada com sucesso.")

In [0]:
# Carregamento da tabela Bronze
df_tipo_veiculo_bronze = spark.table("workspace.bronze.tipo_veiculo")

# Acidentes pertencentes ao município do Rio de Janeiro
df_ids_acidentes_rio = (
    df_acidentes_bronze
    .filter(F.col("codigo_ibge") == "3304557")
    .select("num_acidente")
    .distinct()
)

# Recorte de Tipo Veículo associado aos acidentes do Rio
df_tipo_veiculo_rio = (
    df_tipo_veiculo_bronze
    .join(
        df_ids_acidentes_rio,
        on="num_acidente",
        how="inner"
    )
)

print("Tabela Bronze de tipo_veiculo carregada e recortada para o Rio.")

display(df_tipo_veiculo_rio.limit(20))

In [0]:
df_categorias_veiculo = (
    df_tipo_veiculo_rio
    .groupBy("tipo_veiculo")
    .agg(
        F.sum(F.col("qtde_veiculos").cast("long")).alias("qtde_veiculos")
    )
    .orderBy(F.desc("qtde_veiculos"))
)

display(df_categorias_veiculo)

In [0]:
df_validacao_qtde_veiculos = (
    df_tipo_veiculo_rio
    .filter(
        F.col("qtde_veiculos").isNotNull() &
        F.col("qtde_veiculos").cast("int").isNull()
    )
)

print(
    "Valores de qtde_veiculos não convertíveis para inteiro:",
    df_validacao_qtde_veiculos.count()
)

In [0]:
# Construção da camada Silver de Tipo Veículo
df_tipo_veiculo_silver = (
    df_tipo_veiculo_rio

    # Padronização textual
    .withColumn(
        "tipo_veiculo",
        F.upper(F.trim(F.col("tipo_veiculo")))
    )
    .withColumn(
        "ind_veic_estrangeiro",
        F.upper(F.trim(F.col("ind_veic_estrangeiro")))
    )

    # Tipagem do campo quantitativo
    .withColumn(
        "qtde_veiculos",
        F.col("qtde_veiculos").cast("int")
    )
)

print("DataFrame Silver de tipo_veiculo construído com sucesso.")

In [0]:
display(
    df_tipo_veiculo_silver
    .select(
        "num_acidente",
        "tipo_veiculo",
        "ind_veic_estrangeiro",
        "qtde_veiculos"
    )
    .limit(20)
)

print("\nSchema Silver - Tipo Veículo:")

df_tipo_veiculo_silver.select(
    "num_acidente",
    "tipo_veiculo",
    "ind_veic_estrangeiro",
    "qtde_veiculos"
).printSchema()

In [0]:
# Persistência da tabela Tipo Veículo na camada Silver
(
    df_tipo_veiculo_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.tipo_veiculo")
)

print("Tabela workspace.silver.tipo_veiculo gravada com sucesso.")

In [0]:
# Carregamento da tabela Bronze
df_vitimas_bronze = spark.table("workspace.bronze.vitimas")

# Recorte das vítimas associadas aos acidentes do Rio de Janeiro
df_vitimas_rio = (
    df_vitimas_bronze
    .join(
        df_ids_acidentes_rio,
        on="num_acidente",
        how="inner"
    )
)

print("Tabela Bronze de vítimas carregada e recortada para o Rio.")

display(
    df_vitimas_rio
    .select(
        "num_acidente",
        "data_acidente",
        "uf_acidente",
        "ano_acidente",
        "faixa_idade",
        "genero",
        "tp_envolvido",
        "gravidade_lesao",
        "equip_seguranca",
        "ind_motorista",
        "susp_alcool",
        "qtde_envolvidos",
        "qtde_feridosilesos",
        "qtde_obitos"
    )
    .limit(20)
)

In [0]:
campos_vitimas = [
    "qtde_envolvidos",
    "qtde_feridosilesos",
    "qtde_obitos"
]

for campo in campos_vitimas:
    invalidos = (
        df_vitimas_rio
        .filter(
            F.col(campo).isNotNull() &
            F.col(campo).cast("int").isNull()
        )
        .count()
    )

    print(f"{campo}: {invalidos} valores não convertíveis para inteiro")

In [0]:
# Construção da camada Silver de Vítimas
df_vitimas_silver = (
    df_vitimas_rio

    # Tipagem da data
    .withColumn(
        "data_acidente",
        F.to_date(F.col("data_acidente"), "yyyy-MM-dd")
    )

    # Tipagem dos campos temporais
    .withColumn(
        "ano_acidente",
        F.col("ano_acidente").cast("int")
    )
    .withColumn(
        "mes_acidente",
        F.col("mes_acidente").cast("int")
    )

    # Padronização dos campos categóricos
    .withColumn("faixa_idade", F.upper(F.trim(F.col("faixa_idade"))))
    .withColumn("genero", F.upper(F.trim(F.col("genero"))))
    .withColumn("tp_envolvido", F.upper(F.trim(F.col("tp_envolvido"))))
    .withColumn("gravidade_lesao", F.upper(F.trim(F.col("gravidade_lesao"))))
    .withColumn("equip_seguranca", F.upper(F.trim(F.col("equip_seguranca"))))
    .withColumn("ind_motorista", F.upper(F.trim(F.col("ind_motorista"))))
    .withColumn("susp_alcool", F.upper(F.trim(F.col("susp_alcool"))))

    # Tipagem dos campos quantitativos
    .withColumn("qtde_envolvidos", F.col("qtde_envolvidos").cast("int"))
    .withColumn("qtde_feridosilesos", F.col("qtde_feridosilesos").cast("int"))
    .withColumn("qtde_obitos", F.col("qtde_obitos").cast("int"))
)

print("DataFrame Silver de vítimas construído com sucesso.")

In [0]:
# Validação visual e do schema da tabela Silver de Vítimas
display(
    df_vitimas_silver
    .select(
        "num_acidente",
        "data_acidente",
        "ano_acidente",
        "mes_acidente",
        "faixa_idade",
        "genero",
        "tp_envolvido",
        "gravidade_lesao",
        "qtde_envolvidos",
        "qtde_feridosilesos",
        "qtde_obitos"
    )
    .limit(20)
)

print("\nSchema Silver - Vítimas:")
df_vitimas_silver.select(
    "data_acidente",
    "ano_acidente",
    "mes_acidente",
    "qtde_envolvidos",
    "qtde_feridosilesos",
    "qtde_obitos"
).printSchema()

In [0]:
# Persistência da tabela Vítimas na camada Silver
(
    df_vitimas_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.vitimas")
)

print("Tabela workspace.silver.vitimas gravada com sucesso.")

In [0]:
# Validação final das tabelas persistidas na camada Silver

tabelas_silver = [
    "workspace.silver.acidentes",
    "workspace.silver.localidade",
    "workspace.silver.tipo_veiculo",
    "workspace.silver.vitimas"
]

for tabela in tabelas_silver:
    df = spark.table(tabela)

    print(
        f"{tabela}: "
        f"{df.count():,} registros | "
        f"{len(df.columns)} colunas"
    )